# Day 063 — Exercise 2: check_rate_limit

Rate limiting prevents free-tier abuse and ensures fair resource allocation. Each plan has a daily request allowance. When `usage_count >= limit`, the request is blocked and the API returns `429 Too Many Requests`.

| Plan | Daily limit |
|------|-------------|
| free | 10 |
| pro | 1,000 |
| enterprise | unlimited (∞) |

In [ ]:
FEATURE_MATRIX = {
    "free": {"basic_chat", "view_history"},
    "pro":  {"basic_chat", "view_history", "advanced_chat", "export", "api_access"},
    "enterprise": {"basic_chat", "view_history", "advanced_chat", "export",
                   "api_access", "white_label", "priority_support"},
}

DAILY_LIMITS = {
    "free":       10,
    "pro":        1_000,
    "enterprise": float("inf"),
}


## Task

Implement `check_rate_limit(usage_count, plan) -> tuple[bool, str]`:

- Look up `plan` in `DAILY_LIMITS` (default `0` for unknown plans)
- If `usage_count >= limit`: return `(False, descriptive_message)`
- Otherwise: return `(True, '')`
- `float('inf')` comparisons work correctly: `999_999 >= float('inf')` is `False`

## Your Implementation

In [ ]:
def check_rate_limit(usage_count: int, plan: str) -> tuple[bool, str]:
    """Return (allowed, reason) based on daily usage limits.

    allowed=True, reason=''   — request can proceed
    allowed=False, reason=... — limit reached; include plan name and counts

    Use DAILY_LIMITS to look up the limit for the plan.
    Unknown plan limit defaults to 0 (nothing allowed).
    Enterprise limit is float('inf') — always allowed.
    """
    # TODO: look up limit, compare usage_count, return (bool, str)
    raise NotImplementedError


In [ ]:
def check_rate_limit(usage_count: int, plan: str) -> tuple[bool, str]:
    limit = DAILY_LIMITS.get(plan, 0)
    if usage_count >= limit:
        return False, f"Daily limit reached for {plan!r} plan"
    return True, ""


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # free user under limit
    allowed, msg = check_rate_limit(5, "free")
    assert allowed is True and msg == ""
    score += 1; print("\u2705 free user (5/10) is allowed")

    # free user at limit
    allowed2, msg2 = check_rate_limit(10, "free")
    assert allowed2 is False and isinstance(msg2, str) and len(msg2) > 0
    score += 1; print("\u2705 free user at limit (10/10) is blocked")

    # pro user well under limit
    allowed3, _ = check_rate_limit(500, "pro")
    assert allowed3 is True
    score += 1; print("\u2705 pro user (500/1000) is allowed")

    # pro user at limit
    allowed4, msg4 = check_rate_limit(1000, "pro")
    assert allowed4 is False
    score += 1; print("\u2705 pro user at limit (1000/1000) is blocked")

    # enterprise: always allowed (inf limit)
    allowed5, _ = check_rate_limit(999_999, "enterprise")
    assert allowed5 is True
    score += 1; print("\u2705 enterprise user always allowed (inf limit)")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def check_rate_limit(usage_count: int, plan: str) -> tuple[bool, str]:
    limit = DAILY_LIMITS.get(plan, 0)
    if usage_count >= limit:
        return False, f"Daily limit reached for {plan!r} plan"
    return True, ""
```

**Why `>= limit` not `> limit`?** If the limit is 10, usage_count=10 means the user has already made 10 requests (0 through 9). Request 11 would bring it to 10 completed — but since we check BEFORE incrementing, `usage_count == 10` means the 11th request is arriving. Block it.

</details>